Reserves stress testing 

In [ ]:


# =========================================================
# STRESS TESTING OF SEVERITY PARAMETER UNCERTAINTY
# Uses the already-solved base reserve: base_required_reserve
# Uses base premiums already stored in premium_projection_df
#
# This version includes:
#   1. Base Severity
#   2. P25 Severity  (all parameters set to 25th bootstrap percentile)
#   3. P75 Severity  (all parameters set to 75th bootstrap percentile)
#   4. Best Severity (2.5% / 97.5% corner combo with LOWEST mean)
#   5. Worst Severity (2.5% / 97.5% corner combo with HIGHEST mean)
# =========================================================

# ---------------------------------------------------------
# 2.5% / 97.5% bounds used for Best / Worst corner scenarios
# ---------------------------------------------------------
SEVERITY_PARAMETER_BOUNDS = {
    "cobalt": {
        "dist": "lognormal",
        "mu": (13.0371, 13.0902),
        "sigma": (0.8366, 0.8752),
    },
    "gold": {
        "dist": "logskewnorm",
        "shape": (-1.0710, -0.2655),
        "loc": (17.3875, 17.8099),
        "scale": (0.8903, 1.0573),
    },
    "lithium": {
        "dist": "lognormal",
        "mu": (13.2007, 13.2552),
        "sigma": (0.8637, 0.9021),
    },
    "platinum": {
        "dist": "logskewnorm",
        "shape": (-1.0212, 0.4443),
        "loc": (15.4846, 16.3735),
        "scale": (0.8926, 1.0613),
    },
    "rare earth": {
        "dist": "logskewnorm",
        "shape": (0.9184, 1.4180),
        "loc": (11.4637, 11.6827),
        "scale": (0.9799, 1.1098),
    },
    "supplies": {
        "dist": "logexpweibull",
        "a": (0.4312, 0.7778),
        "c": (1.2883, 2.2817),
        "loc_log": (10.3421, 10.3432),
        "scale_log": (0.9975, 1.3922),
    },
    "titanium": {
        "dist": "logexpweibull",
        "a": (0.4427, 0.6363),
        "c": (2.1361, 2.7283),
        "loc_log": (10.3386, 10.3480),
        "scale_log": (1.6002, 1.8313),
    },
}

# ---------------------------------------------------------
# Direct parameter scenarios from bootstrap percentiles
# ---------------------------------------------------------
P25_SEVERITY_MODELS = {
    "cobalt": {
        "dist": "lognormal",
        "mu": 13.0542,
        "sigma": 0.8501,
    },
    "gold": {
        "dist": "logskewnorm",
        "shape": -0.9024,
        "loc": 17.6314,
        "scale": 0.9596,
    },
    "lithium": {
        "dist": "lognormal",
        "mu": 13.2163,
        "sigma": 0.8755,
    },
    "platinum": {
        "dist": "logskewnorm",
        "shape": -0.8358,
        "loc": 16.1675,
        "scale": 0.9492,
    },
    "rare earth": {
        "dist": "logskewnorm",
        "shape": 1.0882,
        "loc": 11.5294,
        "scale": 1.0268,
    },
    "supplies": {
        "dist": "logexpweibull",
        "a": 0.4765,
        "c": 1.9399,
        "loc_log": 10.3422,
        "scale_log": 1.2582,
    },
    "titanium": {
        "dist": "logexpweibull",
        "a": 0.4923,
        "c": 2.3169,
        "loc_log": 10.3415,
        "scale_log": 1.6748,
    },
}

P75_SEVERITY_MODELS = {
    "cobalt": {
        "dist": "lognormal",
        "mu": 13.0709,
        "sigma": 0.8631,
    },
    "gold": {
        "dist": "logskewnorm",
        "shape": -0.6922,
        "loc": 17.7364,
        "scale": 1.0143,
    },
    "lithium": {
        "dist": "lognormal",
        "mu": 13.2360,
        "sigma": 0.8894,
    },
    "platinum": {
        "dist": "logskewnorm",
        "shape": -0.6064,
        "loc": 16.2927,
        "scale": 1.0089,
    },
    "rare earth": {
        "dist": "logskewnorm",
        "shape": 1.2696,
        "loc": 11.6005,
        "scale": 1.0738,
    },
    "supplies": {
        "dist": "logexpweibull",
        "a": 0.5474,
        "c": 2.1393,
        "loc_log": 10.3425,
        "scale_log": 1.3367,
    },
    "titanium": {
        "dist": "logexpweibull",
        "a": 0.5619,
        "c": 2.5396,
        "loc_log": 10.3436,
        "scale_log": 1.7575,
    },
}


def severity_mean_from_model(model, n_mc=200_000, seed=12345):
    """
    Returns E[X] where X is the severity random variable.
    For lognormal we use the closed-form mean.
    For log-skew-normal and log-exp-weibull we use Monte Carlo.
    """
    dist_name = model["dist"]

    if dist_name == "lognormal":
        return float(np.exp(model["mu"] + 0.5 * model["sigma"] ** 2))

    rng = np.random.default_rng(seed)

    if dist_name == "logskewnorm":
        y = stats.skewnorm.rvs(
            a=model["shape"],
            loc=model["loc"],
            scale=model["scale"],
            size=n_mc,
            random_state=rng,
        )
        return float(np.exp(y).mean())

    if dist_name == "logexpweibull":
        y = stats.exponweib.rvs(
            a=model["a"],
            c=model["c"],
            loc=model["loc_log"],
            scale=model["scale_log"],
            size=n_mc,
            random_state=rng,
        )
        return float(np.exp(y).mean())

    raise ValueError(f"Unknown severity distribution: {dist_name}")


def build_extreme_mean_severity_models(base_models, bounds_dict):
    """
    For each cargo type:
    - Best case: choose 2.5/97.5 corner parameters giving LOWEST mean severity
    - Worst case: choose 2.5/97.5 corner parameters giving HIGHEST mean severity
    """
    best_models = copy.deepcopy(base_models)
    worst_models = copy.deepcopy(base_models)
    summary_rows = []

    for cargo_type, bounds in bounds_dict.items():
        dist_name = bounds["dist"]
        param_names = [k for k in bounds.keys() if k != "dist"]
        corner_values = [bounds[p] for p in param_names]

        best_mean = np.inf
        worst_mean = -np.inf
        best_model = None
        worst_model = None

        for combo in itertools.product(*corner_values):
            candidate = {"dist": dist_name}
            for p, v in zip(param_names, combo):
                candidate[p] = float(v)

            candidate_mean = severity_mean_from_model(candidate)

            if candidate_mean < best_mean:
                best_mean = candidate_mean
                best_model = copy.deepcopy(candidate)

            if candidate_mean > worst_mean:
                worst_mean = candidate_mean
                worst_model = copy.deepcopy(candidate)

        best_models[cargo_type] = best_model
        worst_models[cargo_type] = worst_model

        summary_rows.append({
            "cargo_type": cargo_type,
            "best_mean": best_mean,
            "worst_mean": worst_mean,
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("cargo_type").reset_index(drop=True)
    return best_models, worst_models, summary_df


def build_mean_comparison_table(base_models, p25_models, p75_models, best_models, worst_models):
    rows = []

    for cargo_type in sorted(base_models.keys()):
        rows.append({
            "cargo_type": cargo_type,
            "base_mean": severity_mean_from_model(base_models[cargo_type]),
            "p25_mean": severity_mean_from_model(p25_models[cargo_type]),
            "p75_mean": severity_mean_from_model(p75_models[cargo_type]),
            "best_mean": severity_mean_from_model(best_models[cargo_type]),
            "worst_mean": severity_mean_from_model(worst_models[cargo_type]),
        })

    return pd.DataFrame(rows).sort_values("cargo_type").reset_index(drop=True)


# ---------------------------------------------------------
# Build severity stress scenarios
# ---------------------------------------------------------
BASE_SEVERITY_MODELS = copy.deepcopy(SEVERITY_MODELS)

BEST_SEVERITY_MODELS, WORST_SEVERITY_MODELS, extreme_summary_df = build_extreme_mean_severity_models(
    base_models=BASE_SEVERITY_MODELS,
    bounds_dict=SEVERITY_PARAMETER_BOUNDS,
)

severity_stress_summary_df = build_mean_comparison_table(
    base_models=BASE_SEVERITY_MODELS,
    p25_models=P25_SEVERITY_MODELS,
    p75_models=P75_SEVERITY_MODELS,
    best_models=BEST_SEVERITY_MODELS,
    worst_models=WORST_SEVERITY_MODELS,
)

SEVERITY_STRESS_SCENARIOS = {
    "Base Severity": BASE_SEVERITY_MODELS,
    "P25 Severity": P25_SEVERITY_MODELS,
    "P75 Severity": P75_SEVERITY_MODELS,
    "Best Severity": BEST_SEVERITY_MODELS,
    "Worst Severity": WORST_SEVERITY_MODELS,
}

print("\n=== SEVERITY STRESS SCENARIO SETUP ===")
print(severity_stress_summary_df)

print(f"\nReserve used for stress testing = {base_required_reserve:,.2f}")
print("Premiums remain fixed at the base-case priced premiums.")

# ---------------------------------------------------------
# Run stress test using the reserve already found
# ---------------------------------------------------------
stress_test_rows = []
stress_summary_rows = []

original_severity_models = copy.deepcopy(SEVERITY_MODELS)

for scenario_name, scenario_models in SEVERITY_STRESS_SCENARIOS.items():
    SEVERITY_MODELS = copy.deepcopy(scenario_models)

    scenario_result_df, scenario_worst_annual_ruin = simulate_balance_sheet_paths_with_fixed_initial_reserve(
        case_name=scenario_name,
        case_settings=EXPERIENCE_CASES["Base Experience"],
        pricing_projection_df=premium_projection_df,   # keep base premiums fixed
        n_sims=N_SIMS,
        n_years=N_YEARS,
        seed=RANDOM_SEED + 500,
        initial_reserve=base_required_reserve,
    )

    scenario_output_df = scenario_result_df[
        [
            "year",
            "prob_ruin_in_year",
            "prob_ruin_to_date",
            "mean_total_profit",
            "mean_closing_assets",
        ]
    ].copy()

    scenario_output_df = scenario_output_df.rename(
        columns={
            "prob_ruin_in_year": "probability_of_ruin_in_year",
            "prob_ruin_to_date": "probability_of_ruin_to_date",
            "mean_total_profit": "year_on_year_profit",
            "mean_closing_assets": "expected_end_of_year_reserve",
        }
    )

    scenario_output_df["scenario"] = scenario_name
    scenario_output_df["reserve_used"] = base_required_reserve
    scenario_output_df["cumulative_expected_profit"] = scenario_output_df["year_on_year_profit"].cumsum()

    stress_test_rows.append(scenario_output_df)

    stress_summary_rows.append(
        {
            "scenario": scenario_name,
            "reserve_used": base_required_reserve,
            "max_probability_of_ruin_in_year": scenario_output_df["probability_of_ruin_in_year"].max(),
            "probability_of_ruin_to_year_10": scenario_output_df["probability_of_ruin_to_date"].iloc[-1],
            "total_expected_profit_10_years": scenario_output_df["year_on_year_profit"].sum(),
            "expected_end_of_year_10_reserve": scenario_output_df["expected_end_of_year_reserve"].iloc[-1],
            "passes_1pct_test_every_year": bool(
                scenario_output_df["probability_of_ruin_in_year"].max() <= TARGET_ANNUAL_RUIN_PROB
            ),
        }
    )

SEVERITY_MODELS = original_severity_models

stress_test_df = pd.concat(stress_test_rows, ignore_index=True)
stress_summary_df = pd.DataFrame(stress_summary_rows).sort_values("scenario").reset_index(drop=True)

pd.options.display.float_format = "{:,.4f}".format

print("\n=== STRESS TEST SUMMARY ===")
print(stress_summary_df)

print("\n=== YEARLY STRESS TEST RESULTS ===")
print(
    stress_test_df[
        [
            "scenario",
            "year",
            "reserve_used",
            "probability_of_ruin_in_year",
            "probability_of_ruin_to_date",
            "year_on_year_profit",
            "cumulative_expected_profit",
            "expected_end_of_year_reserve",
        ]
    ]
)

# ---------------------------------------------------------
# Graph 1: Probability of ruin by year
# ---------------------------------------------------------
plt.figure(figsize=(11, 6))
for scenario_name, grp in stress_test_df.groupby("scenario"):
    plt.plot(
        grp["year"],
        grp["probability_of_ruin_in_year"],
        marker="o",
        label=scenario_name,
    )
plt.axhline(TARGET_ANNUAL_RUIN_PROB, linestyle="--", label="1% target")
plt.title("Stress Test: Probability of Ruin in Each Year")
plt.xlabel("Year")
plt.ylabel("Probability of ruin")
plt.legend()
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Graph 2: Year-on-year profit
# ---------------------------------------------------------
plt.figure(figsize=(11, 6))
for scenario_name, grp in stress_test_df.groupby("scenario"):
    plt.plot(
        grp["year"],
        grp["year_on_year_profit"],
        marker="o",
        label=scenario_name,
    )
plt.title("Stress Test: Year-on-Year Profit")
plt.xlabel("Year")
plt.ylabel("Expected profit")
plt.legend()
plt.tight_layout()
plt.show()